In [ ]:
import os
import gc
import torch
import torch.nn as nn
import torch.fft
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, ConcatDataset, random_split
import torch.optim as optim
import torch.nn.functional as F

torch.cuda.empty_cache()
gc.collect()

class FrequencyConvLayer(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.amp_conv = nn.Conv2d(channels, channels, kernel_size=1)
        self.phase_conv = nn.Conv2d(channels, channels, kernel_size=1)

    def forward(self, x):
        x_f32 = x.float()
        freq_repr = torch.fft.rfft2(x_f32, norm="ortho")
        amplitude = torch.abs(freq_repr)
        phase = torch.angle(freq_repr)
        log_amplitude = torch.log(amplitude + 1e-8)
        amp_logits = self.amp_conv(log_amplitude.to(x.dtype))
        amp_mask = torch.sigmoid(amp_logits.float()) * 2.0
        learned_amp = amplitude * amp_mask
        phase_shift = self.phase_conv(phase.to(x.dtype))
        learned_phase = phase + phase_shift.float()
        complex_repr = learned_amp * torch.exp(1j * learned_phase)
        spatial_repr = torch.fft.irfft2(complex_repr, s=x.shape[-2:], norm="ortho")
        return spatial_repr.to(x.dtype)

class CrossModalityMixBlock(nn.Module):
    def __init__(self, spatial_channels, freq_channels):
        super().__init__()
        self.spatial_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(spatial_channels, freq_channels, kernel_size=1),
            nn.ReLU(), nn.Sigmoid()
        )
        self.freq_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(freq_channels, spatial_channels, kernel_size=1),
            nn.ReLU(), nn.Sigmoid()
        )

    def forward(self, spatial_feat, freq_feat):
        freq_gate = self.spatial_attention(spatial_feat)
        spatial_gate = self.freq_attention(freq_feat)
        return spatial_feat + (spatial_feat * spatial_gate), freq_feat + (freq_feat * freq_gate)

class StateOfTheArtHybridDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        self.spatial_features = self.spatial_backbone.features
        
        self.freq_stem = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=7, stride=2, padding=3, bias=False),
            nn.InstanceNorm2d(128, affine=True), nn.ReLU(), nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.freq_stream = FrequencyConvLayer(128)
        
        self.mix_block = CrossModalityMixBlock(spatial_channels=1536, freq_channels=128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        self.classifier = nn.Sequential(
            nn.Linear(1536 + 128, 512), 
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(512, 1)
        )

    def forward(self, x):
        s_feat = self.spatial_features(x)
        f_feat = self.freq_stream(self.freq_stem(x))
        s_feat, f_feat = self.mix_block(s_feat, f_feat)
        
        s_vec = self.global_pool(s_feat).view(s_feat.size(0), -1)
        f_vec = self.global_pool(f_feat).view(f_feat.size(0), -1)
        return self.classifier(torch.cat((s_vec, f_vec), dim=1))

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha 
        self.gamma = gamma 

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss
        return F_loss.mean()

if __name__ == "__main__":
    base_path_primary = "D:/AI_Datasets/ff+/1000_videos"
    base_path_cifake  = "D:/AI_Datasets/cifake"
    base_path_dfdc    = "D:/AI_Datasets/dfdc"

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.3),
        transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_primary = ImageFolder(root=os.path.join(base_path_primary, "train"), transform=train_transform)
    val_primary   = ImageFolder(root=os.path.join(base_path_primary, "validation"), transform=val_transform)

    train_cifake = ImageFolder(root=os.path.join(base_path_cifake, "train"), transform=train_transform)
    full_cifake_test = ImageFolder(root=os.path.join(base_path_cifake, "test"), transform=val_transform)
    c_val_sz = len(full_cifake_test) // 2
    val_cifake, _ = random_split(full_cifake_test, [c_val_sz, len(full_cifake_test)-c_val_sz], generator=torch.Generator().manual_seed(42))

    train_dfdc = ImageFolder(root=os.path.join(base_path_dfdc, "train"), transform=train_transform)
    val_dfdc   = ImageFolder(root=os.path.join(base_path_dfdc, "validation"), transform=val_transform)

    train_dataset = ConcatDataset([train_primary, train_cifake, train_dfdc])
    val_dataset   = ConcatDataset([val_primary, val_cifake, val_dfdc])

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = StateOfTheArtHybridDetector().to(device)

    for i, layer in enumerate(model.spatial_features):
        if i < 6: 
            for param in layer.parameters():
                param.requires_grad = False

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(5):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                predictions = model(images)
                loss = criterion(predictions, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
                
        scheduler.step()
        torch.cuda.empty_cache()
        gc.collect()
        
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).float().unsqueeze(1)
                with torch.amp.autocast('cuda'):
                    predictions = model(images)
                    loss = criterion(predictions, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(predictions) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                
        print(f">> Epoch {epoch+1} Metrics | Train Loss: {running_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Accuracy: {(correct/total)*100:.2f}%")

    save_path = "models/model_v5_sota_efficientnet.pth"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)